# QuVINE: Graph Complexity Analysis and Embedding Demo

This notebook demonstrates:
1. Generating random graphs with known structures
2. Computing comprehensive complexity metrics (including QBioCode)
3. Running QuVINE embedding pipeline
4. Gene/node prioritization task
5. Evaluating results

**Author**: QuVINE Team  
**Date**: 2026

## Setup and Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

# QuVINE imports
from quvine.data import (
    generate_barabasi_albert,
    generate_modular_network,
    generate_graph_with_seeds_and_targets,
    compute_graph_complexity_metrics,
    compute_qbc_complexity_from_laplacian,
    check_qbc_available,
)
from quvine.complexity_pipeline import compute_and_save_complexity

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ Imports successful")
print(f"QBioCode available: {check_qbc_available()}")

## Part 1: Generate Random Graphs

We'll generate different types of random graphs to test embedding algorithms.

In [ ]:
# Set parameters
n_nodes = 200
num_seeds = 20
num_targets = 30
seed = 42

print(f"Generating graphs with {n_nodes} nodes")
print(f"Seeds: {num_seeds}, Targets: {num_targets}")
print("="*60)

### 1.1 Scale-Free Network (Barabási-Albert)

In [ ]:
# Generate scale-free network with seeds and targets
G_ba, seeds_ba, targets_ba = generate_graph_with_seeds_and_targets(
    n=n_nodes,
    num_seeds=num_seeds,
    num_targets=num_targets,
    graph_type='barabasi_albert',
    m=3,
    seed=seed
)

print(f"Scale-Free Network:")
print(f"  Nodes: {G_ba.number_of_nodes()}")
print(f"  Edges: {G_ba.number_of_edges()}")
print(f"  Avg Degree: {2*G_ba.number_of_edges()/G_ba.number_of_nodes():.2f}")
print(f"  Seeds: {len(seeds_ba)}")
print(f"  Targets: {len(targets_ba)}")

### 1.2 Modular Network (Community Structure)

In [ ]:
# Generate modular network
G_mod, seeds_mod, targets_mod = generate_graph_with_seeds_and_targets(
    n=n_nodes,
    num_seeds=num_seeds,
    num_targets=num_targets,
    graph_type='modular',
    num_communities=5,
    p_intra=0.3,
    p_inter=0.01,
    seed=seed
)

print(f"Modular Network:")
print(f"  Nodes: {G_mod.number_of_nodes()}")
print(f"  Edges: {G_mod.number_of_edges()}")
print(f"  Avg Degree: {2*G_mod.number_of_edges()/G_mod.number_of_nodes():.2f}")
print(f"  Seeds: {len(seeds_mod)}")
print(f"  Targets: {len(targets_mod)}")

### 1.3 Visualize Graphs

In [ ]:
def visualize_graph_with_roles(G, seeds, targets, title="Graph"):
    """Visualize graph with seed and target nodes highlighted."""
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Layout
    pos = nx.spring_layout(G, seed=42, k=0.5)
    
    # Node colors
    node_colors = []
    for node in G.nodes():
        if node in seeds:
            node_colors.append('red')
        elif node in targets:
            node_colors.append('green')
        else:
            node_colors.append('lightblue')
    
    # Draw
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, 
                          node_size=50, alpha=0.8, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.2, ax=ax)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red', label=f'Seeds ({len(seeds)})'),
        Patch(facecolor='green', label=f'Targets ({len(targets)})'),
        Patch(facecolor='lightblue', label='Other nodes')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    return fig

# Visualize both graphs
fig1 = visualize_graph_with_roles(G_ba, seeds_ba, targets_ba, 
                                   "Scale-Free Network (Barabási-Albert)")
plt.show()

fig2 = visualize_graph_with_roles(G_mod, seeds_mod, targets_mod, 
                                   "Modular Network (5 Communities)")
plt.show()

## Part 2: Compute Graph Complexity Metrics

Compute comprehensive complexity metrics including spectral properties and QBioCode measures.

### 2.1 Standard Complexity Metrics

In [ ]:
# Compute complexity for both graphs
print("Computing complexity metrics...\n")

complexity_ba = compute_graph_complexity_metrics(G_ba)
complexity_mod = compute_graph_complexity_metrics(G_mod)

# Display results
metrics_to_show = [
    'num_nodes', 'num_edges', 'spectral_gap', 'algebraic_connectivity',
    'spectral_entropy', 'von_neumann_entropy', 'quantum_complexity',
    'centrality_entropy', 'centrality_gini', 'centrality_range'
]

comparison_df = pd.DataFrame({
    'Metric': metrics_to_show,
    'Scale-Free': [complexity_ba[m] for m in metrics_to_show],
    'Modular': [complexity_mod[m] for m in metrics_to_show]
})

print("Complexity Comparison (including Centrality Metrics):")
print(comparison_df.to_string(index=False))
print("\nNote: Higher Gini coefficient indicates more centralized structure")
print("="*60)

### 2.1.1 Fiedler Eigenvalue (Sparse Computation)

In [ ]:
from quvine.data import fiedler_eigenvalue_sparse

print("Computing Fiedler eigenvalue (algebraic connectivity) using sparse methods...\n")

# Compute for both graphs
lambda2_ba, fiedler_ba = fiedler_eigenvalue_sparse(G_ba, normalized=True)
lambda2_mod, fiedler_mod = fiedler_eigenvalue_sparse(G_mod, normalized=True)

print(f"Scale-Free Network:")
print(f"  Fiedler eigenvalue (λ₂): {lambda2_ba:.6f}")
print(f"  Fiedler vector range: [{fiedler_ba.min():.4f}, {fiedler_ba.max():.4f}]")
print(f"\nModular Network:")
print(f"  Fiedler eigenvalue (λ₂): {lambda2_mod:.6f}")
print(f"  Fiedler vector range: [{fiedler_mod.min():.4f}, {fiedler_mod.max():.4f}]")
print(f"\nNote: Fiedler vector can be used for graph partitioning/clustering")
print("="*60)

### 2.2 QBioCode Complexity (if available)

In [ ]:
if check_qbc_available():
    print("Computing QBioCode complexity metrics...\n")
    
    # Compute with mean-centered Laplacian for better complexity metrics
    qbc_ba = compute_qbc_complexity_from_laplacian(
        G_ba, normalized=True, laplacian_method='eigenvectors',
        dataset_name='scale_free', mean_center=True
    )
    
    qbc_mod = compute_qbc_complexity_from_laplacian(
        G_mod, normalized=True, laplacian_method='eigenvectors',
        dataset_name='modular', mean_center=True
    )
    
    # Display key QBC metrics
    qbc_metrics = ['Intrinsic_Dimension', 'Condition number', 'Isomap Reconstruction Error']
    
    qbc_comparison = pd.DataFrame({
        'QBC Metric': qbc_metrics,
        'Scale-Free': [qbc_ba.get(m, 'N/A') for m in qbc_metrics],
        'Modular': [qbc_mod.get(m, 'N/A') for m in qbc_metrics]
    })
    
    print("QBioCode Complexity Comparison (with mean-centered Laplacian):")
    print(qbc_comparison.to_string(index=False))
    print("\nNote: mean_center=True improves manifold complexity estimation")
else:
    print("QBioCode not available. Install from: https://github.com/IBM/QBioCode/")

### 2.3 Visualize Complexity Comparison

In [ ]:
# Plot complexity comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics_to_plot = [
    ('spectral_gap', 'Spectral Gap'),
    ('von_neumann_entropy', 'Von Neumann Entropy'),
    ('quantum_complexity', 'Quantum Complexity')
]

for ax, (metric, title) in zip(axes, metrics_to_plot):
    values = [complexity_ba[metric], complexity_mod[metric]]
    bars = ax.bar(['Scale-Free', 'Modular'], values, 
                   color=['#FF6B6B', '#4ECDC4'], alpha=0.7)
    ax.set_ylabel('Value')
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Part 3: Run QuVINE Embedding Pipeline

We'll run a simplified version of QuVINE to generate embeddings.

### 3.1 Generate Random Walks

In [ ]:
from quvine.walks.rwr import generate_RWR_pagerank_walks
from quvine.embedding.word2vec import corpus_to_embedding

def generate_walks_simple(G, num_walks=10, walk_length=40, restart_prob=0.15, seed=42):
    """Generate random walks for all nodes."""
    rng = np.random.default_rng(seed)
    all_walks = []
    
    for node in G.nodes():
        for _ in range(num_walks):
            walk = [node]
            current = node
            
            for _ in range(walk_length - 1):
                if rng.random() < restart_prob:
                    current = node
                else:
                    neighbors = list(G.neighbors(current))
                    if neighbors:
                        current = rng.choice(neighbors)
                walk.append(current)
            
            all_walks.append([str(n) for n in walk])
    
    return all_walks

print("Generating random walks...")
walks_ba = generate_walks_simple(G_ba, num_walks=10, walk_length=40, seed=seed)
walks_mod = generate_walks_simple(G_mod, num_walks=10, walk_length=40, seed=seed)

print(f"Scale-Free: {len(walks_ba)} walks generated")
print(f"Modular: {len(walks_mod)} walks generated")
print(f"Example walk: {walks_ba[0][:10]}...")

### 3.2 Train Word2Vec Embeddings

In [ ]:
from gensim.models import Word2Vec

def train_embeddings(walks, nodes, embedding_dim=128, window=10, seed=42):
    """Train Word2Vec embeddings from walks."""
    model = Word2Vec(
        sentences=walks,
        vector_size=embedding_dim,
        window=window,
        min_count=1,
        sg=1,  # Skip-gram
        workers=4,
        seed=seed,
        epochs=10
    )
    
    # Extract embeddings in node order
    embeddings = np.zeros((len(nodes), embedding_dim))
    for i, node in enumerate(nodes):
        node_str = str(node)
        if node_str in model.wv:
            embeddings[i] = model.wv[node_str]
    
    return embeddings

print("Training embeddings...")
emb_ba = train_embeddings(walks_ba, list(G_ba.nodes()), embedding_dim=64, seed=seed)
emb_mod = train_embeddings(walks_mod, list(G_mod.nodes()), embedding_dim=64, seed=seed)

print(f"Scale-Free embeddings shape: {emb_ba.shape}")
print(f"Modular embeddings shape: {emb_mod.shape}")

### 3.3 Visualize Embeddings (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

def visualize_embeddings_tsne(embeddings, nodes, seeds, targets, title="Embeddings"):
    """Visualize embeddings using t-SNE."""
    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    emb_2d = tsne.fit_transform(embeddings)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Color by role
    colors = []
    for node in nodes:
        if node in seeds:
            colors.append('red')
        elif node in targets:
            colors.append('green')
        else:
            colors.append('lightgray')
    
    # Plot
    scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], 
                        c=colors, alpha=0.6, s=50)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red', label=f'Seeds ({len(seeds)})'),
        Patch(facecolor='green', label=f'Targets ({len(targets)})'),
        Patch(facecolor='lightgray', label='Other nodes')
    ]
    ax.legend(handles=legend_elements)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1')
    ax.set_ylabel('t-SNE Dimension 2')
    plt.tight_layout()
    return fig

# Visualize both
fig1 = visualize_embeddings_tsne(emb_ba, list(G_ba.nodes()), seeds_ba, targets_ba,
                                  "Scale-Free Network Embeddings (t-SNE)")
plt.show()

fig2 = visualize_embeddings_tsne(emb_mod, list(G_mod.nodes()), seeds_mod, targets_mod,
                                  "Modular Network Embeddings (t-SNE)")
plt.show()

## Part 4: Node/Gene Prioritization Task

Use embeddings to prioritize target nodes based on seed nodes.

### 4.1 Compute Seed Centroid Scores

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_prioritization_scores(embeddings, nodes, seeds):
    """Compute prioritization scores using seed centroid."""
    # Get seed indices
    node_list = list(nodes)
    seed_indices = [node_list.index(s) for s in seeds if s in node_list]
    
    # Compute seed centroid
    seed_embeddings = embeddings[seed_indices]
    centroid = seed_embeddings.mean(axis=0, keepdims=True)
    
    # Compute cosine similarity to centroid
    similarities = cosine_similarity(embeddings, centroid).flatten()
    
    # Create scores dictionary
    scores = {node: sim for node, sim in zip(node_list, similarities)}
    
    return scores

# Compute scores
scores_ba = compute_prioritization_scores(emb_ba, G_ba.nodes(), seeds_ba)
scores_mod = compute_prioritization_scores(emb_mod, G_mod.nodes(), seeds_mod)

print("Prioritization scores computed")
print(f"Score range (Scale-Free): [{min(scores_ba.values()):.3f}, {max(scores_ba.values()):.3f}]")
print(f"Score range (Modular): [{min(scores_mod.values()):.3f}, {max(scores_mod.values()):.3f}]")

### 4.2 Evaluate Target Recovery

In [ ]:
def evaluate_target_recovery(scores, seeds, targets, k_values=[10, 20, 30, 50]):
    """Evaluate how well targets are recovered in top-k predictions."""
    # Sort nodes by score (excluding seeds)
    ranked_nodes = sorted(
        [(node, score) for node, score in scores.items() if node not in seeds],
        key=lambda x: x[1],
        reverse=True
    )
    
    results = []
    for k in k_values:
        top_k = [node for node, _ in ranked_nodes[:k]]
        recovered = len(set(top_k) & set(targets))
        precision = recovered / k
        recall = recovered / len(targets)
        
        results.append({
            'k': k,
            'recovered': recovered,
            'precision': precision,
            'recall': recall
        })
    
    return pd.DataFrame(results)

# Evaluate both graphs
print("Scale-Free Network - Target Recovery:")
eval_ba = evaluate_target_recovery(scores_ba, seeds_ba, targets_ba)
print(eval_ba.to_string(index=False))
print("\n" + "="*60 + "\n")

print("Modular Network - Target Recovery:")
eval_mod = evaluate_target_recovery(scores_mod, seeds_mod, targets_mod)
print(eval_mod.to_string(index=False))

### 4.3 Visualize Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision plot
axes[0].plot(eval_ba['k'], eval_ba['precision'], 'o-', 
             label='Scale-Free', linewidth=2, markersize=8)
axes[0].plot(eval_mod['k'], eval_mod['precision'], 's-', 
             label='Modular', linewidth=2, markersize=8)
axes[0].set_xlabel('k (Top-k predictions)', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title('Precision @ k', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Recall plot
axes[1].plot(eval_ba['k'], eval_ba['recall'], 'o-', 
             label='Scale-Free', linewidth=2, markersize=8)
axes[1].plot(eval_mod['k'], eval_mod['recall'], 's-', 
             label='Modular', linewidth=2, markersize=8)
axes[1].set_xlabel('k (Top-k predictions)', fontsize=12)
axes[1].set_ylabel('Recall', fontsize=12)
axes[1].set_title('Recall @ k', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.4 Precision-Recall Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(eval_ba['recall'], eval_ba['precision'], 'o-', 
        label='Scale-Free', linewidth=2, markersize=8)
ax.plot(eval_mod['recall'], eval_mod['precision'], 's-', 
        label='Modular', linewidth=2, markersize=8)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

## Part 5: Correlation with Complexity

Analyze how graph complexity relates to embedding performance.

In [ ]:
# Create summary
summary = pd.DataFrame({
    'Graph Type': ['Scale-Free', 'Modular'],
    'Quantum Complexity': [
        complexity_ba['quantum_complexity'],
        complexity_mod['quantum_complexity']
    ],
    'Von Neumann Entropy': [
        complexity_ba['von_neumann_entropy'],
        complexity_mod['von_neumann_entropy']
    ],
    'Spectral Gap': [
        complexity_ba['spectral_gap'],
        complexity_mod['spectral_gap']
    ],
    'Precision@20': [
        eval_ba[eval_ba['k']==20]['precision'].values[0],
        eval_mod[eval_mod['k']==20]['precision'].values[0]
    ],
    'Recall@20': [
        eval_ba[eval_ba['k']==20]['recall'].values[0],
        eval_mod[eval_mod['k']==20]['recall'].values[0]
    ]
})

print("\nComplexity vs Performance Summary:")
print("="*80)
print(summary.to_string(index=False))
print("="*80)

## Part 6: Conclusions and Insights

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

# Compare complexities
qc_ba = complexity_ba['quantum_complexity']
qc_mod = complexity_mod['quantum_complexity']

print("\n1. Graph Complexity:")
if qc_ba > qc_mod:
    print(f"   • Scale-Free network has higher quantum complexity ({qc_ba:.3f} vs {qc_mod:.3f})")
    print("   • This suggests quantum walks may provide more benefit for scale-free networks")
else:
    print(f"   • Modular network has higher quantum complexity ({qc_mod:.3f} vs {qc_ba:.3f})")
    print("   • This suggests quantum walks may provide more benefit for modular networks")

# Compare performance
prec_ba = eval_ba[eval_ba['k']==20]['precision'].values[0]
prec_mod = eval_mod[eval_mod['k']==20]['precision'].values[0]

print("\n2. Embedding Performance:")
if prec_ba > prec_mod:
    print(f"   • Scale-Free network shows better target recovery (Precision@20: {prec_ba:.3f} vs {prec_mod:.3f})")
else:
    print(f"   • Modular network shows better target recovery (Precision@20: {prec_mod:.3f} vs {prec_ba:.3f})")

print("\n3. Recommendations:")
if qc_ba > 0.5 or qc_mod > 0.5:
    print("   • High quantum complexity detected - consider using CTQW or DTQW")
else:
    print("   • Moderate complexity - classical random walks (RWR) may be sufficient")

print("\n" + "="*80)

## Summary

This notebook demonstrated:

1. **Random Graph Generation**: Created scale-free and modular networks with designated seed and target nodes
2. **Complexity Analysis**: Computed comprehensive metrics including spectral properties and QBioCode measures
3. **Embedding Generation**: Used random walks and Word2Vec to create node embeddings
4. **Gene Prioritization**: Evaluated target recovery using seed-based scoring
5. **Performance Analysis**: Compared embedding quality across different graph structures

### Next Steps

- Try different graph types (hierarchical, core-periphery, etc.)
- Experiment with quantum walks (CTQW, DTQW)
- Test on real biological networks
- Compare with other embedding methods (Node2Vec, DeepWalk, etc.)
- Use fusion methods to combine multiple embeddings